In [3]:
!pip install --upgrade yfinance


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import requests, os
from datetime import date
import pandas as pd

### Import via Companies MarketCap

In [5]:
# --- Configuração ---
URL = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
OUTPUT_DIR = "../data/raw/"
FILE_PREFIX = "top_companies_revenue"
# --------------------

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{today}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Tamanho: {len(response.content):,} bytes")


Arquivo salvo em: ../data/raw/top_companies_revenue_20261010.csv
Tamanho: 774,243 bytes


### Padronização e Ajuste de Nomenclatura do Dataset

In [6]:
import glob
files = sorted(glob.glob("../data/raw/top_companies_revenue_*.csv"))
raw_data_file = files[-1]

print(f"Arquivo mais recente encontrado: {raw_data_file}")

import pandas as pd
df = pd.read_csv(raw_data_file)

std_columns = {
    "Rank": "rank",
    "Name": "company_name",
    "Symbol": "ticker_symbol",
    "revenue_ttm": "revenue_ttm",
    "price (USD)": "price_usd",
    "country": "company_country",
}
df = df.rename(columns=std_columns)


from bs4 import BeautifulSoup
def clean_html(text):
    if pd.isna(text):
        return text
    return BeautifulSoup(text, "html.parser").get_text()

df["company_name"] = df["company_name"].apply(clean_html)

#check for non unique tickers and print the ones that are not unique 
non_unique_tickers = df[df.duplicated(subset=["ticker_symbol"], keep=False)]
if not non_unique_tickers.empty:
    print("Non-unique tickers found:")
    print(non_unique_tickers[["ticker_symbol", "company_name"]])

display(df.head(1))

Arquivo mais recente encontrado: ../data/raw\top_companies_revenue_20261010.csv


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country
0,1,Amazon,AMZN,742776000000,262.43,United States


### Limpar Dataset seguindo escopo estabelecido

In [7]:

# retirar empresas com receita negativa ou nula
df = df[df["revenue_ttm"] > 0]

# listar linhas que estao duplicadas no subset de colunas "company_name" e "revenue_ttm"
duplicated_rows = df[df.duplicated(subset=["company_name", "revenue_ttm"], keep=False)]
print("Linhas duplicadas:")
print(duplicated_rows)

# drop_duplicates remove linhas repetidas; subset define o que conta como "igual"
# Usar nome + receita evita apagar homônimos diferentes (Castellum, First Bancorp)
df = df.drop_duplicates(subset=["company_name", "revenue_ttm"])

Linhas duplicadas:
     rank   company_name ticker_symbol  revenue_ttm  price_usd company_country
189   190  SK Innovation     096775.KS  62996237095    55.5274     South Korea
190   191  SK Innovation     096770.KS  62996237095   119.9240     South Korea


### Informações Preliminares do Dataset

In [8]:
import pandas as pd

PIB = 118_350_166_039_396  # Banco Mundial, 2025, USD correntes

# numero de empresas com receita positiva
num_companies = len(df)
print(f"Número de empresas com receita positiva: {num_companies}")

# Receita total após a limpeza
total = df["revenue_ttm"].sum()

print(f"Total: {total/1e12:.2f} tri")
print(f"% do PIB: {total/PIB*100:.1f}%")

# 1) Soma do top 500 por receita: deve ficar perto dos ~41,7 tri da Fortune 500
#    nlargest(500, coluna) devolve as 500 linhas com maior valor na coluna
top500 = df.nlargest(500, "revenue_ttm")["revenue_ttm"].sum()
print(f"Top 500: {top500/1e12:.1f} tri")

# 2) Os 20 maiores: confira à mão com a Fortune (Walmart, Amazon, State Grid, etc.)
print(df.nlargest(20, "revenue_ttm")[["company_name", "revenue_ttm"]])

# 3) Duplicidades: nomes repetidos podem indicar ADR/dupla listagem
#    duplicated(keep=False) marca TODAS as linhas repetidas, não só a segunda
dups = df[df.duplicated(subset="company_name", keep=False)].sort_values("company_name")
print(len(dups))

# 4) Peso dos maiores: se poucas linhas explicam muito do total, olhe-as primeiro
print(df["revenue_ttm"].describe())

Número de empresas com receita positiva: 10635
Total: 66.87 tri
% do PIB: 56.5%
Top 500: 39.2 tri
                            company_name   revenue_ttm
0                                 Amazon  742776000000
1                                Walmart  735840000000
2                           Saudi Aramco  487870749993
3                                  Apple  466823000000
4                           UnitedHealth  450129000000
5                      Alphabet (Google)  445867000000
6                                Sinopec  444749587530
7                     Berkshire Hathaway  420276000000
8                             PetroChina  419928669654
9                             CVS Health  415086000000
10                              McKesson  410983000000
11                            Volkswagen  374700624687
12                           Exxon Mobil  361060000000
13                               Cencora  332771323000
14                                Toyota  332273722835
15                    

### Enriquecimento com Yahoo Finance

In [9]:
import os, time
import pandas as pd
from functools import partial
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from IPython.display import display, clear_output

import importlib
import src.yf_pipeline as yfp   # "src" é subpasta da pasta atual do notebook

importlib.reload(yfp)                    # recarrega o .py editado sem reiniciar o kernel

# ---------------- CONFIG ----------------
OUTPUT_PATH = "../data/interim/01_data_after_yfinance.csv"   # checkpoint
CACHE_PATH  = "../data/interim/yf_cache.csv"
CHUNK_SIZE, WAIT_TIME, MAX_WORKERS = 100, 5, 10
PAUSA_RATE_LIMIT, JITTER_MAX = 60, 3.0

# ---------------- 1. PREPARAR ----------------
yfp.garantir_colunas(df)                     # cria colunas yf_* que faltarem (in-place)
if os.path.exists(OUTPUT_PATH):              # retoma do checkpoint
    yfp.aplicar_resultados(df, pd.read_csv(OUTPUT_PATH))
cache = yfp.carregar_cache(CACHE_PATH)       # dict em memória
print(f"Cache: {len(cache)} tickers")

# ---------------- 2. PENDENTES ----------------
pendentes = yfp.listar_pendentes(df, reprocessar_not_found=True, ordem_reversa=True)
print(f"Total: {len(df)} | Pendentes: {len(pendentes)}")

# ---------------- 3. LOOP (visível aqui) ----------------
# partial "fixa" cache e parâmetros: executor.map só aceita funções de 1 argumento
resolver = partial(yfp.resolver_ticker, cache=cache,
                   pausa_segundos=PAUSA_RATE_LIMIT, jitter_max=JITTER_MAX)

for i in tqdm(range(0, len(pendentes), CHUNK_SIZE), desc="Lotes"):
    clear_output()
    chunk = pendentes[i:i + CHUNK_SIZE]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        resultados = list(ex.map(resolver, chunk))      # 1 chamada por ticker

    yfp.aplicar_resultados(df, resultados)              # grava no df, por ticker
    yfp.atualizar_cache(cache, resultados)              # sucessos novos entram no dict
    yfp.salvar_cache(cache, CACHE_PATH)                 # dict -> CSV
    yfp.salvar_csv(df, OUTPUT_PATH)                     # checkpoint por lote

    # só espera se algum ticker do lote realmente foi ao Yahoo
    if any(r["origem"] == "yahoo" for r in resultados):
        time.sleep(WAIT_TIME)

# ---------------- 4. AJUSTES E VISUALIZAÇÃO ----------------

# fazer limpeza pós resolver: se o yf_status é "success" mas o website está ausente, então o ticker não foi resolvido de fato
df = pd.read_csv(OUTPUT_PATH)

# Máscara 1: linhas onde o yf_status não foi sucesso
sucesso = df["yf_status"] == "success"
# Máscara 2: website ausente (NaN real) ou string vazia
website_vazio = df["website"].isna() | (df["website"] == "")

# Combina as duas condições com & (E lógico bit a bit — para Series,
# use sempre & / | / ~ em vez de and/or/not, que não funcionam elemento a elemento)
mask = sucesso & website_vazio
df.loc[mask, "yf_status"] = "error: ticker not resolved"
# opcional: conferir quantas linhas foram afetadas
print(mask.sum())
df.to_csv(OUTPUT_PATH, index=False)


clear_output()
print(f"Total de tickers: {len(df)} | Resolvidos: {len(df[df['yf_status'] == 'success'])} | Não Resolvidos: {len(df[df['yf_status'] == 'error: ticker not resolved'])} | Cache: {len(cache)}")

display(df.head(10))

Total de tickers: 10635 | Resolvidos: 10286 | Não Resolvidos: 348 | Cache: 10845


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
0,1,Amazon,AMZN,742776000000,262.430000,United States,https://www.amazon.com,"Amazon.com, Inc. engages in the retail sale of...",Consumer Cyclical,Internet Retail,United States,Seattle,success
1,2,Walmart,WMT,735840000000,111.350000,United States,https://www.stock.walmart.com,Walmart Inc. engages in the operation of retai...,Consumer Defensive,Discount Stores,United States,Bentonville,success
2,3,Saudi Aramco,2222.SR,487870749993,6.855590,Saudi Arabia,https://www.aramco.com,Saudi Arabian Oil Company operates as an integ...,Energy,Oil & Gas Integrated,Saudi Arabia,Dhahran,success
3,4,Apple,AAPL,466823000000,336.640000,United States,https://www.apple.com,"Apple Inc. designs, manufactures, and markets ...",Technology,Consumer Electronics,United States,Cupertino,success
4,5,UnitedHealth,UNH,450129000000,379.300000,United States,https://www.unitedhealthgroup.com,UnitedHealth Group Incorporated operates as a ...,Healthcare,Healthcare Plans,United States,Eden Prairie,success
5,6,Alphabet (Google),GOOG,445867000000,347.860000,United States,https://abc.xyz,Alphabet Inc. offers various products and plat...,Communication Services,Internet Content & Information,United States,Mountain View,success
6,7,Sinopec,600028.SS,444749587530,0.806645,China,https://www.sinopec.com/listco,"China Petroleum & Chemical Corporation, an ene...",Energy,Oil & Gas Integrated,China,Beijing,success
7,8,Berkshire Hathaway,BRK-B,420276000000,515.620000,United States,https://www.berkshirehathaway.com,"Berkshire Hathaway Inc., together with its sub...",Financial Services,Insurance - Diversified,United States,Omaha,success
8,9,PetroChina,0857.HK,419928669654,1.252610,China,https://www.petrochina.com.cn,"PetroChina Company Limited, together with its ...",Energy,Oil & Gas Integrated,China,Beijing,success
9,10,CVS Health,CVS,415086000000,86.160000,United States,https://www.cvshealth.com,CVS Health Corporation provides health solutio...,Healthcare,Healthcare Plans,United States,Woonsocket,success


### Análise sobre lacunas de informação

In [11]:
import pandas as pd

df = pd.read_csv(OUTPUT_PATH)

# Status que você quer medir (ajuste se o texto for diferente)
STATUS_ERRO = "error: ticker not resolved"

total_geral = df["revenue_ttm"].sum()

for n in (500, 1000, 2000, 5000, df.shape[0]):

    topn = df.nlargest(n, "revenue_ttm")
    nao_resolvidas = topn[topn["yf_status"] == STATUS_ERRO]

    # Métricas de cobertura em quantidade e em receita
    receita_topn = topn["revenue_ttm"].sum()
    receita_nr = nao_resolvidas["revenue_ttm"].sum()

    print(f"Top {n}:")
    print(f"  Não resolvidas: {len(nao_resolvidas)} de {n} ({len(nao_resolvidas)/n*100:.1f}%)")
    print(f"  Receita do top {n}: {receita_topn/1e12:.2f} tri "
          f"({receita_topn/total_geral*100:.1f}% do total)")
    print(f"  Receita não resolvida: {receita_nr/1e12:.3f} tri "
          f"({receita_nr/receita_topn*100:.2f}% do top {n})")

      # print limhas de cada empresa faltante, com todas as colunas, para inspeção manual
    display(nao_resolvidas)
    

Top 500:
  Não resolvidas: 3 de 500 (0.6%)
  Receita do top 500: 39.24 tri (58.7% do total)
  Receita não resolvida: 0.120 tri (0.31% do top 500)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
267,269,Tata Motors,TATAMOTORS.NS,46922406926,4.51722,India,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
297,299,Phoenix Group,PHNX.L,42882249061,10.41780,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
419,421,Michelin,ML.PA,30533561783,36.94760,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 1000:
  Não resolvidas: 5 de 1000 (0.5%)
  Receita do top 1000: 48.61 tri (72.7% do total)
  Receita não resolvida: 0.156 tri (0.32% do top 1000)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
267,269,Tata Motors,TATAMOTORS.NS,46922406926,4.51722,India,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
297,299,Phoenix Group,PHNX.L,42882249061,10.41780,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
419,421,Michelin,ML.PA,30533561783,36.94760,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
643,645,Fiserv,FISV,20866000000,45.63000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
940,942,Dongfeng Motor,0489.HK,14551321587,1.21850,China,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 2000:
  Não resolvidas: 10 de 2000 (0.5%)
  Receita do top 2000: 57.29 tri (85.7% do total)
  Receita não resolvida: 0.202 tri (0.35% do top 2000)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
267,269,Tata Motors,TATAMOTORS.NS,46922406926,4.51722,India,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
297,299,Phoenix Group,PHNX.L,42882249061,10.41780,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
419,421,Michelin,ML.PA,30533561783,36.94760,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
643,645,Fiserv,FISV,20866000000,45.63000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
940,942,Dongfeng Motor,0489.HK,14551321587,1.21850,China,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1161,1163,Grupo Elektra,XEKT.MC,11317736000,20.24580,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1246,1248,Hino Motors,7205.T,10354593406,2.42961,Japan,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1366,1368,QVC Group,QVCGA,9231000000,0.34000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1568,1570,International Holding Company,IHC.AE,7747610218,97.47000,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1648,1650,Seven Group Holdings\r\n (SGH),SGH.AX,7320175531,26.45950,Australia,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 5000:
  Não resolvidas: 67 de 5000 (1.3%)
  Receita do top 5000: 65.10 tri (97.4% do total)
  Receita não resolvida: 0.343 tri (0.53% do top 5000)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
267,269,Tata Motors,TATAMOTORS.NS,46922406926,4.51722,India,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
297,299,Phoenix Group,PHNX.L,42882249061,10.41780,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
419,421,Michelin,ML.PA,30533561783,36.94760,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
643,645,Fiserv,FISV,20866000000,45.63000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
940,942,Dongfeng Motor,0489.HK,14551321587,1.21850,China,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,4797,Nilfisk Holding,NLFSK.CO,1147368156,21.92390,Denmark,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
4813,4815,Bed Bath & Beyond,BBBY,1139531000,4.35000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
4832,4834,Shoe Carnival,SCVL,1128339000,17.43000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
4895,4897,Itaú CorpBanca,ITCLY,1087155900,3.10000,Chile,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 10635:
  Não resolvidas: 348 de 10635 (3.3%)
  Receita do top 10635: 66.87 tri (100.0% do total)
  Receita não resolvida: 0.400 tri (0.60% do top 10635)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
267,269,Tata Motors,TATAMOTORS.NS,46922406926,4.51722,India,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
297,299,Phoenix Group,PHNX.L,42882249061,10.41780,United Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
419,421,Michelin,ML.PA,30533561783,36.94760,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
643,645,Fiserv,FISV,20866000000,45.63000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
940,942,Dongfeng Motor,0489.HK,14551321587,1.21850,China,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10595,10597,Advent Technologies,ADN,49000,0.91850,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
10596,10598,Zoom Telephonics,MINM,45118,3.61000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
10604,10606,Forza X1,FRZA,18559,0.20000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
10606,10608,Lucy Scientific Discovery,LSDI,16732,0.51390,Canada,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
